# একটি VLM প্রশিক্ষণ: ধাপে ধাপে Pipeline

একটি VLM প্রশিক্ষণ -- staged pipeline, catastrophic forgetting, এবং loss masking, সবই একটি প্রকৃত (ক্ষুদ্র) autoregressive model-এ পরিমাপ করা।

কোনো download নেই, কোনো pretrained weight নেই। CPU-তে কয়েক মিনিট।

Setup-টি একটি toy যতটা পারে ততটাই প্রকৃত সমস্যার কাছাকাছি:

- একটি ছোট causal Transformer LM-কে একটি শুধু-টেক্সট task-এ **PRETRAIN** করা হয়: key/value জোড়ার একটি sequence, তারপর একটি query key, উত্তর = সেই key-এর value। এটি "একটি language model যা ইতিমধ্যেই কাজ করে"-এর প্রতিনিধি, আর এটি আমাদের একটি টেক্সট ক্ষমতা দেয় যা vision যোগ করার আগে ও পরে **MEASURE** করা যায়।
- এরপর vision যোগ করা হয়: একই উত্তর, কিন্তু key/value জোড়াগুলো text token-এর বদলে vision token হিসেবে আসে (প্রতি object-এ একটি)। এটি task-এর VQA দিক।
- দুটি task-ই model, vocabulary এবং LM head ভাগ করে, তাই language model-কে ক্ষতিগ্রস্ত করে এমন যেকোনো কিছু সঙ্গে সঙ্গে হারানো text accuracy হিসেবে দেখা দেয় -- বাস্তবে catastrophic forgetting ঠিক এভাবেই ধরা হয়।

Experiment-গুলো:

1. ছয়টি training schedule (projector-only, staged, joint, replay-mixed, high-LR), **দুটি** task-এই score করা। Forgetting কলামটিই আসল কথা।
2. Loss masking: প্রতিটি token-এর উপর LM loss বনাম শুধু উত্তরের উপর।
3. Vision tower freeze করা বনাম train করা।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান (CPU-তে মোট কয়েক মিনিট)।

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- vocabulary -------------------------------------------------------------
N_VALUES = 8            # answer token: id 0..7
N_KEYS = 12             # text key:     id 8..19
N_SHAPES = 12           # image shape:  id 20..31
Q_TEXT, Q_IMG = 32, 33  # "প্রশ্নের উত্তর দাও" marker
VOCAB = 34

KEY0, SHAPE0 = N_VALUES, N_VALUES + N_KEYS

N_PAIRS = 4             # প্রতি text sample-এ key/value জোড়া
N_OBJECTS = 4           # প্রতি image-এ object (vision token)
N_FILLER = 6            # image prompt-এ "instruction phrasing" token --
                        # random, অননুমেয়, এবং তাই untrainable
D_VISION = 32
D_TOWER_OUT = 4         # tower-এর output width: একটি কঠোর information bottleneck
D_MODEL = 64

shape_code = torch.randn(N_SHAPES, D_VISION)
value_code = torch.randn(N_VALUES, D_VISION)

## 0. Data: একই lookup task, একবার টেক্সটে, একবার pixel-এ

দুটি batch generator: `text_batch` শুধু-টেক্সট key/value sequence তৈরি করে, আর `image_batch` একই ধরনের তথ্য vision token হিসেবে দেয়, সাথে random "instruction" filler token-সহ একটি prompt। এখানে কোনো training চলে না — কেবল পরের সেকশনগুলোর জন্য data প্রস্তুতি।

In [ ]:
# ---------------------------------------------------------------------------
# 0. Data: একই lookup task, একবার টেক্সটে, একবার pixel-এ
# ---------------------------------------------------------------------------

def text_batch(b):
    """[k1 v1 k2 v2 ... Q_TEXT kq a] -- বিশুদ্ধ টেক্সট, কোনো image নেই।"""
    keys = torch.stack([torch.randperm(N_KEYS)[:N_PAIRS] for _ in range(b)]) + KEY0
    vals = torch.randint(0, N_VALUES, (b, N_PAIRS))
    slot = torch.randint(0, N_PAIRS, (b,))
    qk = keys[torch.arange(b), slot]
    ans = vals[torch.arange(b), slot]
    seq = torch.stack([torch.stack([keys[:, i], vals[:, i]]) for i in range(N_PAIRS)])
    seq = seq.permute(2, 0, 1).reshape(b, 2 * N_PAIRS)          # k,v,k,v... interleave
    seq = torch.cat([seq,
                     torch.full((b, 1), Q_TEXT),
                     qk.unsqueeze(1),
                     ans.unsqueeze(1)], dim=1)
    return seq, ans


def image_batch(b):
    """vision token (প্রতি object-এ একটি) + [Q_IMG shape_q a]।"""
    shapes = torch.stack([torch.randperm(N_SHAPES)[:N_OBJECTS] for _ in range(b)])
    vals = torch.randint(0, N_VALUES, (b, N_OBJECTS))
    vision = shape_code[shapes] + value_code[vals]
    vision = vision + 0.2 * torch.randn_like(vision)
    slot = torch.randint(0, N_OBJECTS, (b,))
    ans = vals[torch.arange(b), slot]
    qs = shapes[torch.arange(b), slot] + SHAPE0
    # একটি randomly phrased instruction ("please describe...", "tell me...") --
    # এখানে শুধু random token। এটি context, এবং এটি অনুমেয় নয়।
    filler = torch.randint(KEY0, KEY0 + N_KEYS, (b, N_FILLER))
    seq = torch.cat([torch.full((b, 1), Q_IMG), filler, qs.unsqueeze(1), ans.unsqueeze(1)], dim=1)
    return vision, seq, ans

## 1. একটি ন্যূনতম causal Transformer + একটি projector

একটি model, দুটি input পথ। `Block` ও `causal_mask` একটি সাধারণ decoder তৈরি করে; `VLM` class-এ LM অংশ (যা একটি প্রকৃত pipeline pretrained অবস্থায় পায়) এবং নতুন `vision_tower` ও `projector` আছে। `lm_loss` next-token cross-entropy গণনা করে (ঐচ্ছিকভাবে শুধু উত্তরের উপর), আর `accuracy` দুটি task-এ model-কে score করে।

In [ ]:
# ---------------------------------------------------------------------------
# 1. একটি ন্যূনতম causal Transformer + একটি projector
# ---------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, 4, batch_first=True)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.ff = nn.Sequential(nn.Linear(D_MODEL, 4 * D_MODEL), nn.GELU(),
                                nn.Linear(4 * D_MODEL, D_MODEL))

    def forward(self, x, mask):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, need_weights=False)[0]
        return x + self.ff(self.ln2(x))


def causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)


class VLM(nn.Module):
    """একটি model, দুটি input পথ। `lm` হল সেই অংশ যা থেকে একটি প্রকৃত pipeline
    শুরু করত (pretrained); `projector` ও `vision_tower` নতুন।"""

    def __init__(self, n_layers=2):
        super().__init__()
        self.tok_embed = nn.Embedding(VOCAB, D_MODEL)
        self.pos_embed = nn.Parameter(torch.randn(1, 32, D_MODEL) * 0.02)
        self.blocks = nn.ModuleList([Block() for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB)
        # "নতুন" module, শুধু-টেক্সট pretraining-এ অনুপস্থিত।
        # Tower-এর output সরু (D_TOWER_OUT << D_VISION): একটি প্রকৃত vision
        # encoder-এর মতো, এটি image-এর সবকিছু সামনে পাঠাতে পারে না, আর এটি
        # কোন (WHICH) তথ্য রাখে তা ঠিক করেছে তার নিজস্ব pretraining -- এখানে, তার
        # frozen random initialization।
        self.vision_tower = nn.Linear(D_VISION, D_TOWER_OUT, bias=False)
        self.projector = nn.Sequential(nn.Linear(D_TOWER_OUT, D_MODEL), nn.GELU(),
                                       nn.Linear(D_MODEL, D_MODEL))

    def lm_params(self):
        mods = [self.tok_embed, *self.blocks, self.ln_f, self.head]
        ps = [p for m in mods for p in m.parameters()] + [self.pos_embed]
        return ps

    def forward(self, seq, vision=None):
        x = self.tok_embed(seq)
        if vision is not None:
            vis = self.projector(self.vision_tower(vision))
            x = torch.cat([vis, x], dim=1)
        x = x + self.pos_embed[:, : x.shape[1]]
        mask = causal_mask(x.shape[1])
        for blk in self.blocks:
            x = blk(x, mask)
        return self.head(self.ln_f(x))                      # (B, L, VOCAB)


def lm_loss(model, seq, vision=None, answer_only=True):
    """Next-token cross-entropy। `answer_only` উত্তর ছাড়া প্রতিটি position mask
    করে দেয় -- অর্থাৎ prompt হল context, training target নয় (SFT)।"""
    logits = model(seq, vision)
    n_vis = 0 if vision is None else vision.shape[1]
    # Position t থেকে token t+1 predict করা। Text position-গুলো prefix-এর পরে শুরু হয়।
    pred = logits[:, n_vis: -1]                             # (B, L_text-1, V)
    target = seq[:, 1:]                                     # (B, L_text-1)
    if answer_only:
        return F.cross_entropy(pred[:, -1], target[:, -1])
    return F.cross_entropy(pred.reshape(-1, VOCAB), target.reshape(-1))


@torch.no_grad()
def accuracy(model, kind, n=1500):
    if kind == "text":
        seq, ans = text_batch(n)
        logits = model(seq)
        pred = logits[:, -2].argmax(-1)                     # উত্তরের আগের position
    else:
        vision, seq, ans = image_batch(n)
        logits = model(seq, vision)
        pred = logits[:, -2].argmax(-1)
    return (pred == ans).float().mean().item()

## 2. Stage 0: শুধু টেক্সটে language model pretrain করা

`pretrain_lm` শুধু LM parameter-গুলোকে text-only lookup task-এ train করে। ফলাফল `base` হল আমাদের "pretrained LLM" — নিচের প্রতিটি experiment ঠিক এই weight থেকে শুরু হয়, তাই table-গুলোর প্রতিটি পার্থক্য কেবল training schedule-এর কারণে। এই cell text ও image accuracy এবং chance-level print করে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Stage 0: শুধু টেক্সটে language model pretrain করা
# ---------------------------------------------------------------------------

def pretrain_lm(steps=1200, bs=64, lr=2e-3):
    torch.manual_seed(1)
    model = VLM()
    opt = torch.optim.Adam(model.lm_params(), lr=lr)
    for _ in range(steps):
        seq, _ = text_batch(bs)
        loss = lm_loss(model, seq)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return model


print("=" * 78)
print("STAGE 0: pretrain the LM on the TEXT-ONLY lookup task")
print("=" * 78)
base = pretrain_lm()
base_text_acc = accuracy(base, "text")
print(f"text-task accuracy after text-only pretraining : {base_text_acc:.1%}")
print(f"image-task accuracy (vision is untrained noise): {accuracy(base, 'image'):.1%}")
print(f"chance on both tasks                           : {1 / N_VALUES:.1%}")
print()
print("This is our 'pretrained LLM'. Everything below starts from these exact")
print("weights, so every difference in the tables is caused by the training")
print("schedule and nothing else.")
print()

## 3. Multimodal training schedule-গুলো

`train_schedule` `base`-এর একটি copy নিয়ে stage 1 (শুধু connector, LM frozen) এবং stage 2 (ঐচ্ছিকভাবে LM unfreeze, ঐচ্ছিক text replay) চালায়। এরপর ছয়টি schedule চালিয়ে প্রতিটির image accuracy, text accuracy এবং forgetting (মূল text accuracy থেকে পতন) একটি table-এ print করা হয়, সাথে প্রতিটি সারির ব্যাখ্যা।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Multimodal training schedule-গুলো
# ---------------------------------------------------------------------------

import copy


def train_schedule(name, stage1_steps, stage2_steps, unfreeze_lm, lr_stage2,
                   text_replay=0.0, bs=64, lr_stage1=2e-3, freeze_tower=True):
    torch.manual_seed(2)
    model = copy.deepcopy(base)

    # --- Stage 1: alignment। শুধু connector train হয়; LM নড়তে পারে না।
    if stage1_steps:
        params = list(model.projector.parameters())
        if not freeze_tower:
            params += list(model.vision_tower.parameters())
        opt = torch.optim.Adam(params, lr=lr_stage1)
        for _ in range(stage1_steps):
            vision, seq, _ = image_batch(bs)
            loss = lm_loss(model, seq, vision)
            opt.zero_grad()
            loss.backward()
            opt.step()
    acc_after_s1 = accuracy(model, "image") if stage1_steps else float("nan")

    # --- Stage 2: instruction tuning। ঐচ্ছিকভাবে LM unfreeze করা।
    if stage2_steps:
        params = list(model.projector.parameters())
        if not freeze_tower:
            params += list(model.vision_tower.parameters())
        if unfreeze_lm:
            params += model.lm_params()
        opt = torch.optim.Adam(params, lr=lr_stage2)
        for _ in range(stage2_steps):
            vision, seq, _ = image_batch(bs)
            loss = lm_loss(model, seq, vision)
            if text_replay > 0:                     # text-only data আবার মিশিয়ে দেওয়া
                tseq, _ = text_batch(max(1, int(bs * text_replay)))
                loss = loss + lm_loss(model, tseq)
            opt.zero_grad()
            loss.backward()
            opt.step()

    return {
        "name": name,
        "img": accuracy(model, "image"),
        "txt": accuracy(model, "text"),
        "s1": acc_after_s1,
    }


print("=" * 78)
print("1. SIX SCHEDULES, SCORED ON BOTH TASKS")
print("=" * 78)
print("'text' is the ORIGINAL text-only ability, re-measured after multimodal")
print(f"training. It started at {base_text_acc:.1%}. Any drop is forgetting.")
print()

schedules = [
    # name,                             s1,   s2,  unfreeze, lr_s2, replay
    ("stage 1 only (projector, frozen LM)", 700, 0, False, 0.0, 0.0),
    ("stage 1 -> stage 2, LM frozen", 700, 700, False, 1e-3, 0.0),
    ("stage 1 -> stage 2, LM unfrozen", 700, 700, True, 2e-4, 0.0),
    ("stage 1 -> stage 2, LM unfrozen, HIGH lr", 700, 700, True, 2e-3, 0.0),
    ("stage 1 -> stage 2 + 50% text replay", 700, 700, True, 2e-4, 0.5),
    ("NO stage 1: unfreeze everything at once", 0, 1400, True, 2e-3, 0.0),
]

print(f"{'schedule':<44} {'image':>7} {'text':>7} {'forgetting':>11}")
rows = []
for name, s1, s2, unf, lr2, rep in schedules:
    r = train_schedule(name, s1, s2, unf, lr2, rep)
    rows.append(r)
    forget = base_text_acc - r["txt"]
    print(f"{name:<44} {r['img']:>6.1%} {r['txt']:>6.1%} {forget:>10.1%}")

print()
print("What the rows say:")
print()
print("* Freezing the LM makes forgetting IMPOSSIBLE by construction -- the text")
print("  weights never move, so the text score cannot change. That is the whole")
print("  appeal of stage 1: it is a safe way to teach the connector where to put")
print("  visual features, using cheap caption-style data. Note how far short of")
print("  the task it lands on its own, though: a projector alone has to make")
print("  visual features legible to an LM that has never been asked to read")
print("  them, and that ceiling is why stage 2 exists.")
print()
print("* Unfreezing the LM at a SMALL learning rate buys image accuracy at a")
print("  modest cost in text ability. Unfreezing at the same learning rate used")
print("  for the connector wrecks the text task: the multimodal data contains no")
print("  text-only examples, so nothing is holding those weights in place.")
print()
print("* Text replay -- mixing the original text-only data back into the")
print("  multimodal stage -- is the standard fix, and it works here for the same")
print("  reason it works in production: the objective now includes the ability")
print("  you are trying not to lose.")
print()
print("* Skipping stage 1 and unfreezing everything at once trains a randomly")
print("  initialized connector and a working LM with one learning rate. The")
print("  connector's early output is noise, and the LM adapts to noise before")
print("  the connector becomes useful -- which is exactly the instability the")
print("  two-stage recipe was invented to avoid.")
print()

## 4. Loss masking: prompt token target হিসেবে বনাম context হিসেবে

`train_masking` একই model দুইবার train করে — একবার loss শুধু উত্তরের উপর (standard SFT), একবার sequence-এর প্রতিটি token-এর উপর — এবং দুটোর image ও text accuracy তুলনা করে। অননুমেয় prompt token-এর উপর loss দিলে gradient noise-এ খরচ হয় এবং আসল signal পাতলা হয়ে যায়।

In [ ]:
# ---------------------------------------------------------------------------
# 4. Loss masking: prompt token target হিসেবে বনাম context হিসেবে
# ---------------------------------------------------------------------------

def train_masking(answer_only, steps=500, bs=64, lr=1e-3):
    torch.manual_seed(2)
    model = copy.deepcopy(base)
    opt = torch.optim.Adam(list(model.projector.parameters()) + model.lm_params(), lr=lr)
    for _ in range(steps):
        vision, seq, _ = image_batch(bs)
        loss = lm_loss(model, seq, vision, answer_only=answer_only)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return accuracy(model, "image"), accuracy(model, "text")


print("=" * 78)
print("2. LOSS MASKING: is the prompt a training target or just context?")
print("=" * 78)
img_masked, txt_masked = train_masking(answer_only=True)
img_all, txt_all = train_masking(answer_only=False)
print(f"{'loss computed over':<42} {'image acc':>10} {'text acc':>9}")
print(f"{'the answer only (standard SFT)':<42} {img_masked:>9.1%} {txt_masked:>8.1%}")
print(f"{'every token in the sequence':<42} {img_all:>9.1%} {txt_all:>8.1%}")
print()
print("The prompt here contains a randomly chosen query shape, which is")
print("fundamentally unpredictable -- no amount of capacity can lower that part")
print("of the loss. Training on it spends gradient on noise and dilutes the")
print("signal from the one token that matters. Masking the prompt (Phase 05")
print("Lesson 4's recipe, unchanged for images) is not a small detail; on a")
print("real VLM whose prompts include hundreds of vision tokens, the answer is a")
print("tiny fraction of the sequence and the dilution is proportionally worse.")
print()

## 5. Vision tower-ও কি train করা উচিত?

একই staged schedule (50% text replay সহ) দুইবার চালানো হয় — একবার vision tower frozen, একবার trainable — এবং image ও text accuracy তুলনা করা হয়। Tower unfreeze করলে তার নিজস্ব pretraining যা ফেলে দিয়েছিল তা পুনরুদ্ধার করা যায়, কিন্তু প্রকৃত model-এ এর ঝুঁকিও আছে।

In [ ]:
# ---------------------------------------------------------------------------
# 5. Vision tower-ও কি train করা উচিত?
# ---------------------------------------------------------------------------

print("=" * 78)
print("3. FREEZING THE VISION TOWER vs. TRAINING IT")
print("=" * 78)
frozen = train_schedule("frozen tower", 700, 700, True, 2e-4, 0.5, freeze_tower=True)
unfrozen = train_schedule("trained tower", 700, 700, True, 2e-4, 0.5, freeze_tower=False)
print(f"{'vision tower':<30} {'image':>7} {'text':>7}")
print(f"{'frozen':<30} {frozen['img']:>6.1%} {frozen['txt']:>6.1%}")
print(f"{'trained (unfrozen)':<30} {unfrozen['img']:>6.1%} {unfrozen['txt']:>6.1%}")
print()
print("Unfreezing the tower gives the pipeline a way to fix what the tower's own")
print("pretraining objective threw away (Lesson 2 section 4) -- but on a real")
print("model it also risks destroying features that took hundreds of GPU-years")
print("of contrastive training to acquire, on a fine-tuning set orders of")
print("magnitude smaller. Current practice: freeze during alignment, then")
print("unfreeze late at a much lower learning rate than the rest of the model,")
print("if at all.")